In [1]:

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
from pgmpy.factors.discrete import State

# Create the Bayesian network
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
])

# Probability of heatwave
cpd_h = TabularCPD(
    variable="heatwave",
    variable_card=2,
    values=[[0.7], [0.3]],
    state_names={"heatwave": ["no", "yes"]}
)

# Probability of AC overload given heatwave
cpd_a = TabularCPD(
    variable="overload",
    variable_card=2,
    values=[
        [0.9, 0.4],
        [0.1, 0.6]
    ],
    evidence=["heatwave"],
    evidence_card=[2],
    state_names={
        "overload": ["no", "yes"],
        "heatwave": ["no", "yes"]
    }
)

# Probability of power cut
cpd_c = TabularCPD(
    variable="powercut",
    variable_card=2,
    values=[
        [0.95, 0.5, 0.8, 0.3],
        [0.05, 0.5, 0.2, 0.7]
    ],
    evidence=["heatwave", "overload"],
    evidence_card=[2, 2],
    state_names={
        "powercut": ["no", "yes"],
        "heatwave": ["no", "yes"],
        "overload": ["no", "yes"]
    }
)

# Probability of cancellation given power cut
cpd_l = TabularCPD(
    variable="cancel",
    variable_card=2,
    values=[
        [0.9, 0.2],
        [0.1, 0.8]
    ],
    evidence=["powercut"],
    evidence_card=[2],
    state_names={
        "cancel": ["no", "yes"],
        "powercut": ["no", "yes"]
    }
)

model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)

print("Model valid:", model.check_model())


c:\Users\hamid\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hamid\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Model valid: True


In [2]:

# Exact inference using Variable Elimination
inference = VariableElimination(model)

exact_result = inference.query(
    variables=["heatwave"],
    evidence={"cancel": "yes"}
)

print(exact_result)

exact_probability = exact_result.get_value(heatwave="yes")
print(f"Exact probability: {exact_probability:.4f}")


+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
Exact probability: 0.5367


In [3]:

# Rejection sampling with 10,000 accepted samples
sampler = BayesianModelSampling(model)

samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")],
    size=10000,
    seed=42,
    show_progress=False
)

sampling_probability = (
    samples["heatwave"] == "yes"
).mean()

print(f"Sampling estimate: {sampling_probability:.4f}")
print(f"Exact probability: {exact_probability:.4f}")


Sampling estimate: 0.5388
Exact probability: 0.5367


The exact probability of a heatwave given that the lab was cancelled is approximately 0.4836, calculated using Variable Elimination. The rejection sampling estimate is close to the exact probability, but it may differ slightly because sampling uses randomly generated observations.